# SafeMod - Jigsaw evaluation (Colab)
Runs directly from the GitHub repository.

- **Path A (no GPU, ~1 min):** evaluate from the released predictions file `outputs/jigsaw_scoring/jigsaw_predictions.csv`. Run sections 1 and A.
- **Path B (GPU, ~15 min):** re-score the 63,978 labeled test comments with Detoxify 0.5.4 `original`, then evaluate. Set *Runtime > Change runtime type > T4 GPU* and run sections 1 and B.

## 1. Code

In [ ]:
!git clone --depth 1 https://github.com/zeeshanali-ow/safemod /content/safemod
%cd /content/safemod/safemod
!pip install -q -r requirements.txt

## A. Evaluate from the released predictions (no GPU)

In [ ]:
!python jigsaw/eval_jigsaw.py --pred ../outputs/jigsaw_scoring/jigsaw_predictions.csv --out jigsaw_eval_regenerated
!python tools/compare_outputs.py ../outputs/jigsaw_eval jigsaw_eval_regenerated

## B. Re-score from the raw data (GPU)
### B1. Install the scoring model
The published predictions were produced with Detoxify **0.5.4**.

In [ ]:
!nvidia-smi -L
!pip install -q detoxify==0.5.4

### B2. Data: `test.csv` and `test_labels.csv`
First, while signed in to Kaggle, open https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/rules and accept the competition rules (one-time).

**Option 1 - Kaggle API (recommended):** on kaggle.com/settings, under *API*, click **Generate New Token** and copy the token. Run the next cell and paste it when asked (it stays hidden). If you instead have a legacy `kaggle.json`, see the comment inside the cell.

In [ ]:
!pip install -q --upgrade "kaggle>=1.8"
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Paste your Kaggle API token: ')
# Legacy alternative: upload kaggle.json, then
# os.makedirs('/root/.kaggle', exist_ok=True); os.replace('kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge -p data
!cd data && unzip -o -q jigsaw-toxic-comment-classification-challenge.zip && unzip -o -q test.csv.zip && unzip -o -q test_labels.csv.zip
!ls -la data

**Option 2 - manual:** download `test.csv.zip` and `test_labels.csv.zip` from the competition's *Data* tab, upload them to `/content/safemod/safemod/data/` and run:
```
!cd data && unzip -o -q test.csv.zip && unzip -o -q test_labels.csv.zip
```

### B3. (Optional) keep progress on Google Drive
If Colab disconnects, re-running B4 resumes where it stopped, but only if the output folder survives. Mount Drive and use the Drive path as `OUT`.

In [ ]:
OUT = 'jigsaw_outputs'
# from google.colab import drive; drive.mount('/content/drive'); OUT = '/content/drive/MyDrive/safemod_jigsaw'

### B4. Score the test set (GPU)

In [ ]:
!python jigsaw/score_jigsaw.py --data data --out {OUT} --batch-size 64

### B5. Evaluate and verify

In [ ]:
!python jigsaw/eval_jigsaw.py --pred {OUT}/jigsaw_predictions.csv

In [ ]:
import pandas as pd, json, hashlib, importlib.metadata
print('detoxify:', importlib.metadata.version('detoxify'))
h = hashlib.sha256(open(f'{OUT}/jigsaw_predictions.csv','rb').read()).hexdigest()
print('predictions sha256:', h)
print('identical to the released predictions file:', h == '0e9522cc56e283ed24c56dd82957605fcea7a1a1ea8038ba8a7c236941512a71')
print(json.dumps(json.load(open(f'{OUT}/scoring_manifest.json')), indent=1)[:1500])
print(pd.read_csv(f'{OUT}/eval/checks.csv').to_string(index=False))

In [ ]:
!zip -r -q jigsaw_outputs.zip {OUT}
from google.colab import files; files.download('jigsaw_outputs.zip')